# System Dependencies

To get started with Unstructured.io, we need a few system wide dependencies.

## Poppler (poppler-utils)

Handles pdf processing. It's a library that can extract text, images, metadata from PDFs.
Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)

Optical Character Recognition (OCR) engine. When you have scanned documents, images with texts or PDFs that are essentially pictures, tesseract reads the texts from these images and converts it into machine readable texts.

## libmagic

File type detection library. It identifies what type of file you are dealing with (PDF, word doc, image etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [6]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-openai
%pip install -Uq python_dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

#Langchain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

/Users/pge7181/Desktop/demo/RAG/RAG/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import os 

api_key = os.getenv("OPENAI_API_KEY")

In [3]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"

In [4]:
def partition_document(file_path: str):
    """ Extract elements from PDF using unstructured"""
    print(f"Partitioning Documents: {file_path}")

    elements = partition_pdf(
        filename=file_path, # Path to your pdf file
        strategy="hi_res", # Use the most accurate but slower processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML and not jumbled texts
        extract_image_block_types=["Image"], # Grab images found in the pdf
        extract_image_block_to_payload=True # Store images as base64 data you can usually use 
    )

    print(f"Extracted {len(elements)} elements")
    return elements

In [5]:
file_path = "/Users/pge7181/Desktop/demo/RAG/RAG/MULTI-MODAL-RAG/docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

Partitioning Documents: /Users/pge7181/Desktop/demo/RAG/RAG/MULTI-MODAL-RAG/docs/attention-is-all-you-need.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1228.44it/s, Materializing param=model.query_position_embeddings.weight]                            


Extracted 187 elements


In [6]:
len(elements)

187

In [7]:
elements

In [8]:
# All types of different atomic elements we see from unstructured

set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [9]:
elements[30].to_dict()

{'type': 'NarrativeText',
 'element_id': '8302e32ada8c89a8124febeed7047522',
 'text': 'End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [34].',
 'metadata': {'detection_class_prob': 0.9427498579025269,
  'coordinates': {'points': ((np.float64(519.5213012695312),
     np.float64(2643.77490234375)),
    (np.float64(519.5213012695312), np.float64(2779.924560546875)),
    (np.float64(2473.49755859375), np.float64(2779.924560546875)),
    (np.float64(2473.49755859375), np.float64(2643.77490234375))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-23T19:19:31',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 2,
  'file_directory': '/Users/pge7181/Desktop/demo/RAG/RAG/MULTI-MODAL-RAG/docs',
  'filename': 'attention-is-all-you-need.pdf',
  'parent

In [10]:
# Gather all the images

images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images")

images[0].to_dict()

Found 6 images


{'type': 'Image',
 'element_id': 'dddb4d23c61fd226df643d0f3f9a0eb2',
 'text': 'Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | —-Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, SE a, Positional CY Encoding ® Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding',
 'metadata': {'detection_class_prob': 0.8292749524116516,
  'coordinates': {'points': ((np.float64(932.3425903320312),
     np.float64(399.51641845703125)),
    (np.float64(932.3425903320312), np.float64(1908.3768310546875)),
    (np.float64(2014.68798828125), np.float64(1908.3768310546875)),
    (np.float64(2014.68798828125), np.float64(399.51641845703125))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-23T19:19:31',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type

In [11]:
# Gather all the tables
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

Found 4 tables


{'type': 'Table',
 'element_id': 'ec1a0bb601e0e52fd1775c2498fd57d0',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n? - d) O(1) O(1) Recurrent O(n - d?) O(n) O(n) Convolutional O(k-n-d?) O(1) O(log, (7)) Self-Attention (restricted) O(r-n-d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9282333254814148,
  'coordinates': {'points': ((np.float64(561.3707885742188),
     np.float64(550.4574584960938)),
    (np.float64(561.3707885742188), np.float64(908.8244018554688)),
    (np.float64(2395.140380859375), np.float64(908.8244018554688)),
    (np.float64(2395.140380859375), np.float64(550.4574584960938))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-23T19:19:31',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n? - d)</td>

In [12]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("Creating smart chunks ...")

    chunks = chunk_by_title(
        elements, # the parsed pdf elements from previous steps
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # merge tiny chunks under 500 chars with neighbors
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [13]:
chunks = create_chunks_by_title(elements)

Creating smart chunks ...
Created 25 chunks


In [14]:
# view all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [15]:
chunks[2].to_dict()

{'type': 'CompositeElement',
 'element_id': 'b92f7fc2-bc69-43b7-b135-b9a4a866bf1a',
 'text': '1 Introduction\n\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].\n\nRecurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states h,, as a function of the previous hidden state h,_; and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across ex

In [16]:
chunks[4].metadata.orig_elements

In [17]:
chunks[11].metadata.orig_elements

In [18]:
# See the contents of the table

chunks[11].metadata.orig_elements[-1].to_dict()

{'type': 'Table',
 'element_id': 'ec1a0bb601e0e52fd1775c2498fd57d0',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n? - d) O(1) O(1) Recurrent O(n - d?) O(n) O(n) Convolutional O(k-n-d?) O(1) O(log, (7)) Self-Attention (restricted) O(r-n-d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9282333254814148,
  'coordinates': {'points': ((np.float64(561.3707885742188),
     np.float64(550.4574584960938)),
    (np.float64(561.3707885742188), np.float64(908.8244018554688)),
    (np.float64(2395.140380859375), np.float64(908.8244018554688)),
    (np.float64(2395.140380859375), np.float64(550.4574584960938))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-23T19:19:31',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n? - d)</td>

In [19]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/25
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     ❌ AI summary failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
     → AI summary created successfully
     → Enhanced content preview: 3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35

In [20]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "2023\\n\\n1706.03762v7 [cs.CL] 2 Aug\\n\\narXiv\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani* Google Brain avaswani@google.com\\n\\nNoam Shazeer* Google Brain noam@google.com\\n\\nNiki Parmar* Google Research nikip@google.com\\n\\nJakob Uszkoreit* Google Research usz@google.com\\n\\nLlion\\n\\nJones* Google Research llion@google.com\\n\\nAidan N. Gomez* \' University of Toronto aidan@cs.toronto.edu\\n\\nLukasz Kaiser* Google Brain lukaszkaiser@google.com", "tables_html": [], "images_base64": []}'}, page_content="2023\n\n1706.03762v7 [cs.CL] 2 Aug\n\narXiv\n\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\nAttention Is All Y

In [21]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean json format"""
    export_data=[]

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i+1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content","{}"))
            }
        }

        export_data.append(chunk_data)

    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"Exported {len(export_data)} chunks to {filename}")

    return export_data


# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

Exported 25 chunks to chunks_export.json


In [26]:
def create_vector_store(documents, persist_directory="../db/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in chromaDB")

    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small")

    # Create ChromaDB vector Store

    print("---Creating vector store---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )

    print("--- Finished creating vector store ---")

    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)


Creating embeddings and storing in chromaDB
---Creating vector store---


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [27]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/_4/fd6pfxgx479cx75bjsv999hh0000gn/T/ipykernel_14511/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1417.32it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def create_vector_store(documents, persist_directory="../db/chroma_db"):

    print("Creating embeddings and storing in chromaDB")

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("---Creating vector store---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory
    )

    print("--- Finished creating vector store ---")

    return vectorstore

In [29]:
# Create the vector store
db = create_vector_store(processed_chunks)

Creating embeddings and storing in chromaDB


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1436.38it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---Creating vector store---
--- Finished creating vector store ---


In [31]:
# After your retreival
query = "What are the two main components of the Transformer Architecture?"
retriever = db.as_retriever(search_kwargs={"k":3})
chunks = retriever.invoke(query)

# Export to json
export_chunks_to_json(chunks, "rag_results.json")

Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': '6.2. Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n>We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectiv... [Contains 1 table(s)]',
  'metadata': {'original_content': {'raw_text': '6.2. Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n>We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.\n\nTable 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair encodi

In [32]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [ ]:
db = run_complete_ingestion_pipeline("./docs/attention-is-all-you-need.pdf")

🚀 Starting RAG Ingestion Pipeline
Partitioning Documents: ./docs/attention-is-all-you-need.pdf
Extracted 187 elements
Creating smart chunks ...
Created 25 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/25
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     ❌ AI summary failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read t

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1424.04it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---Creating vector store---
--- Finished creating vector store ---
🎉 Pipeline completed successfully!


🚀 Starting RAG Ingestion Pipeline
Partitioning Documents: ./docs/attention-is-all-you-need.pdf


KeyboardInterrupt: 

In [44]:
# Query the vector store
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

❌ Answer generation failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Sorry, I encountered an error while generating the answer.
